# N2 · long-horizon: loop-with-hook 救回 early-stop (配套 L05)

一个 6 步任务, 模型会在第 2 步**错误地收工** (early-stopping 病)。看 hook 如何拦截它、
靠**文件系统 state** 跨窗口接力, 最终完成。全程 OTel trace。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"
sys.path.insert(0, str(SRC))
import provider, compaction, long_horizon, otel_trace, harness_eval
print("src 组件就绪 (默认 MockProvider, 无需 API key)")

src 组件就绪 (默认 MockProvider, 无需 API key)


In [2]:
import tempfile
from long_horizon import run_long_horizon, demo_setup
from compaction import Compactor
from otel_trace import Tracer

work = tempfile.mkdtemp()
provider_, goal, tools, store, goal_met = demo_setup(work, total_steps=6, early_stop_at=2)
tracer = Tracer()

res = run_long_horizon(provider_, goal, tools, store, goal_met,
                       compactor=Compactor(max_tokens=600), tracer=tracer,
                       max_windows=6, hook=True)

print("目标:", goal)
print("成功:", res.success, "| 窗口数:", res.n_windows, "| 总步数:", res.total_steps)
print("被拦截的 early-stop:", [w.index for w in res.windows if w.stop_intercepted])
print("最终文件系统 state:", {k:res.final_state[k] for k in ('progress','goal_total')})

目标: 完成 6 个子步骤
成功: True | 窗口数: 2 | 总步数: 6
被拦截的 early-stop: [0]
最终文件系统 state: {'progress': 6, 'goal_total': 6}


## OTel span 树: window ⊃ reasoning ⊃ tool

In [3]:
print(tracer.render())
print("\n聚合 stats:", tracer.stats())

▭ window-0 (Δ9)
  ◆ reason@w0.s1 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w0.s2 (Δ3)
    → tool:do_step (Δ1)
▭ window-1 (Δ17)
  ◆ reason@w1.s1 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w1.s2 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w1.s3 (Δ3)
    → tool:do_step (Δ1)
  ◆ reason@w1.s4 (Δ3)
    → tool:do_step (Δ1)

聚合 stats: {'window': {'count': 2, 'duration': 26}, 'reasoning': {'count': 6, 'duration': 18}, 'tool': {'count': 6, 'duration': 6}}


## 对照: 关掉 hook, 同样的任务会失败

In [4]:
work2 = tempfile.mkdtemp()
p2, g2, t2, s2, gm2 = demo_setup(work2, total_steps=6, early_stop_at=2)
res_nohook = run_long_horizon(p2, g2, t2, s2, gm2, hook=False, max_windows=6)
print("hook=False → 成功:", res_nohook.success, "| aborted_early:", res_nohook.aborted_early,
      "| 进度只到:", res_nohook.final_state.get('progress'))

hook=False → 成功: False | aborted_early: True | 进度只到: 2


**收获**: 同一个会 early-stop 的模型, **有 hook → 成功 (跨 2 窗口), 无 hook → 失败**。差别全在 harness。文件系统是跨窗口记忆的载体。